# 面试问题：怎样从零设计 ReAct Agent 循环，并防止死循环和成本失控？

**一句话回答**：把 Agent 实现为有界状态机：模型根据目标和可信状态提出结构化 action，宿主校验/授权后执行工具，把 observation 标记为外部数据，再进入下一步；达到完成条件、需要人工、预算/截止时间、重复状态或不可恢复错误时终止。规划质量与执行安全必须分离。

本 Notebook 用可控搜索环境实现 Thought/Action/Observation 轨迹、预算、cycle detector、重规划和轨迹 grader，不调用 Agent 框架或真实模型。

In [ ]:
from dataclasses import dataclass, field  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。

SEED107=10701  # 计算并保存当前步骤的中间状态。
KNOWLEDGE107={"订单A":"状态=已发货;物流单=ZX9","ZX9":"位置=上海;预计=明天","退款规则":"已发货订单需拒收后退款"}  # 计算并保存当前步骤的中间状态。
assert len(KNOWLEDGE107)==3  # 用受控断言验证关键不变量。
assert "ZX9" in KNOWLEDGE107["订单A"]  # 用受控断言验证关键不变量。
assert SEED107==10701  # 用受控断言验证关键不变量。

## 1. 显式状态比自由文本历史更可靠

状态包含目标、步骤、token/工具/时间预算、已知事实、最近 action、错误和终止原因。模型可生成 thought 供调试，但宿主不能把自由文本当控制流。每次转移只允许从预定义状态进入下一状态。

In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class AgentState107:  # 定义承载本节状态与行为的数据结构。
    goal:str; max_steps:int=6; max_tool_calls:int=4; step:int=0; tool_calls:int=0; facts:dict=field(default_factory=dict); trace:list=field(default_factory=list); status:str="running"; stop_reason:str=""  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.goal or self.max_steps<=0 or self.max_tool_calls<0: raise ValueError("state_contract")  # 按当前条件选择后续控制路径。
state107=AgentState107("查询订单A何时送达")  # 计算并保存当前步骤的中间状态。
assert state107.status=="running" and state107.step==0  # 用受控断言验证关键不变量。
assert state107.facts=={} and state107.trace==[]  # 用受控断言验证关键不变量。
try: AgentState107("",0); raise AssertionError("bad state accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="state_contract"  # 捕获预期异常并验证失败分支。

## 2. Action 使用小而严格的语法

这里仅允许 `search(key)` 与 `finish(answer)`。真实系统使用 JSON schema；未知动作、空参数和超长参数直接拒绝，并把可修复错误作为结构化 observation 返回有限次数。禁止让模型拼接任意代码。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Action107: kind:str; argument:str  # 定义承载本节状态与行为的数据结构。
def validate_action107(action):  # 定义本节可复用的核心函数。
    if not isinstance(action,Action107) or action.kind not in {"search","finish"}: return False,"unknown_action"  # 按当前条件选择后续控制路径。
    if not action.argument or len(action.argument)>200: return False,"argument_contract"  # 按当前条件选择后续控制路径。
    return True,"valid"  # 返回当前分支计算出的结果。
assert validate_action107(Action107("search","订单A"))==(True,"valid")  # 用受控断言验证关键不变量。
assert validate_action107(Action107("shell","ls"))==(False,"unknown_action")  # 用受控断言验证关键不变量。
assert validate_action107(Action107("finish",""))==(False,"argument_contract")  # 用受控断言验证关键不变量。

## 3. Observation 是不可信数据，不是新系统指令

工具返回网页、邮件或数据库文本，可能包含“忽略之前规则”等注入。宿主保留来源、时间、权限和截断信息，把内容放入 data channel；下一轮 prompt 明确只能提取事实。工具异常按类型处理，不能伪装成正常 observation。

In [ ]:
def execute107(action):  # 定义本节可复用的核心函数。
    ok,reason=validate_action107(action)  # 计算并保存当前步骤的中间状态。
    if not ok: return {"ok":False,"error":reason}  # 按当前条件选择后续控制路径。
    if action.kind=="finish": return {"ok":True,"final":action.argument}  # 按当前条件选择后续控制路径。
    if action.argument not in KNOWLEDGE107: return {"ok":False,"error":"not_found"}  # 按当前条件选择后续控制路径。
    return {"ok":True,"source":"kb-v1","data":KNOWLEDGE107[action.argument],"untrusted":True}  # 返回当前分支计算出的结果。
obs107=execute107(Action107("search","订单A"))  # 计算并保存当前步骤的中间状态。
assert obs107["ok"] and obs107["untrusted"]  # 用受控断言验证关键不变量。
assert obs107["source"]=="kb-v1"  # 用受控断言验证关键不变量。
assert execute107(Action107("search","未知"))["error"]=="not_found"  # 用受控断言验证关键不变量。

## 4. 用脚本 Policy 代替模型，验证循环本身

测试 Agent 框架时应把模型替换为确定性 policy，构造成功、错误、越权和循环轨迹。下面的 policy 先查订单，再从 observation 提取物流号，查物流，最后回答。它只用于状态机 oracle，不代表真实语言理解。

In [ ]:
def policy107(state):  # 定义本节可复用的核心函数。
    if "订单A" not in state.facts: return Action107("search","订单A")  # 按当前条件选择后续控制路径。
    if "ZX9" not in state.facts: return Action107("search","ZX9")  # 按当前条件选择后续控制路径。
    return Action107("finish",f"订单A已发货，{state.facts['ZX9']}")  # 返回当前分支计算出的结果。
probe_state107=AgentState107("g")  # 计算并保存当前步骤的中间状态。
assert policy107(probe_state107)==Action107("search","订单A")  # 用受控断言验证关键不变量。
probe_state107.facts["订单A"]=KNOWLEDGE107["订单A"]; assert policy107(probe_state107)==Action107("search","ZX9")  # 计算并保存当前步骤的中间状态。
probe_state107.facts["ZX9"]=KNOWLEDGE107["ZX9"]; assert policy107(probe_state107).kind=="finish"  # 计算并保存当前步骤的中间状态。

## 5. 主循环统一管理预算与状态转移

每轮先检查预算，再 proposal/validate/execute/observe；只有工具执行才增加 tool_calls，所有模型轮次都增加 step。finish 经过最终输出校验后结束。状态机使超时、取消和恢复都能落在确定边界。

In [ ]:
def run_agent107(state,policy):  # 定义本节可复用的核心函数。
    while state.status=="running":  # 在终止条件满足前持续推进状态。
        if state.step>=state.max_steps: state.status="stopped"; state.stop_reason="step_budget"; break  # 按当前条件选择后续控制路径。
        action=policy(state); state.step+=1; event={"step":state.step,"action":action}  # 计算并保存当前步骤的中间状态。
        if action.kind=="search":  # 按当前条件选择后续控制路径。
            if state.tool_calls>=state.max_tool_calls: state.status="stopped"; state.stop_reason="tool_budget"; break  # 按当前条件选择后续控制路径。
            state.tool_calls+=1  # 计算并保存当前步骤的中间状态。
        obs=execute107(action); event["observation"]=obs; state.trace.append(event)  # 计算并保存当前步骤的中间状态。
        if not obs["ok"]: state.status="stopped"; state.stop_reason=obs["error"]  # 按当前条件选择后续控制路径。
        elif action.kind=="finish": state.status="completed"; state.stop_reason="finished"; state.facts["answer"]=obs["final"]  # 按当前条件选择后续控制路径。
        else: state.facts[action.argument]=obs["data"]  # 计算并保存当前步骤的中间状态。
    return state  # 返回当前分支计算出的结果。
completed107=run_agent107(AgentState107("查询订单A何时送达"),policy107)  # 计算并保存当前步骤的中间状态。
assert completed107.status=="completed" and completed107.stop_reason=="finished"  # 用受控断言验证关键不变量。
assert completed107.step==3 and completed107.tool_calls==2  # 用受控断言验证关键不变量。
assert "明天" in completed107.facts["answer"]  # 用受控断言验证关键不变量。

## 6. Cycle detector 不能只比较整段字符串

模型可能反复查询同一资源、在两个工具间摆动或换措辞重复。对规范化 `(action, args, relevant_state)` 做指纹，并设置连续重复/窗口频次阈值；触发后可重规划一次，再失败则停止或人工升级。

In [ ]:
def action_fp107(action): return hashlib.sha256(json.dumps({"kind":action.kind,"argument":action.argument.strip().lower()},sort_keys=True).encode()).hexdigest()  # 定义本节可复用的核心函数。
def has_cycle107(actions,window=4,max_same=2):  # 定义本节可复用的核心函数。
    recent=[action_fp107(a) for a in actions[-window:]]; return any(recent.count(x)>max_same for x in set(recent))  # 计算并保存当前步骤的中间状态。
repeated107=[Action107("search","未知")]*3; alternating107=[Action107("search","A"),Action107("search","B"),Action107("search","A"),Action107("search","B")]  # 计算并保存当前步骤的中间状态。
assert has_cycle107(repeated107)  # 用受控断言验证关键不变量。
assert not has_cycle107(alternating107,max_same=2)  # 用受控断言验证关键不变量。
assert action_fp107(Action107("search"," A "))==action_fp107(Action107("search","a"))  # 用受控断言验证关键不变量。

## 7. 错误分类决定 retry、replan 还是 stop

timeout/429 可在总 deadline 内重试；参数 schema 错误可让模型修一次；not_found 应改变查询或拒答；unauthorized 永不通过重试解决；副作用结果未知需查幂等账本。统一错误策略避免模型自行无限尝试。

In [ ]:
ERROR_POLICY107={"timeout":"retry","rate_limited":"retry","argument_contract":"repair_once","not_found":"replan","unauthorized":"stop","unknown_write_result":"reconcile"}  # 计算并保存当前步骤的中间状态。
def error_decision107(error,retries_left):  # 定义本节可复用的核心函数。
    action=ERROR_POLICY107.get(error,"stop")  # 计算并保存当前步骤的中间状态。
    if action=="retry" and retries_left<=0: return "stop"  # 按当前条件选择后续控制路径。
    return action  # 返回当前分支计算出的结果。
assert error_decision107("timeout",1)=="retry"  # 用受控断言验证关键不变量。
assert error_decision107("timeout",0)=="stop"  # 用受控断言验证关键不变量。
assert error_decision107("unauthorized",9)=="stop" and error_decision107("unknown_write_result",0)=="reconcile"  # 用受控断言验证关键不变量。

## 8. 评测 task outcome、轨迹、安全与效率

成功率不能只看 final 字符串。检查是否找到必需事实、是否调用禁止工具、步骤/成本、是否正确拒答，以及同类失败是否可归因。线上记录模型/工具版本、每步延迟和 stop reason，并对高风险 action 做人工审批。

In [ ]:
def grade_agent107(state):  # 定义本节可复用的核心函数。
    required={"订单A","ZX9"}; return {"task_success":state.status=="completed" and required<=set(state.facts),"safe":all(e["action"].kind in {"search","finish"} for e in state.trace),"steps":state.step,"tool_calls":state.tool_calls,"stop":state.stop_reason}  # 计算并保存当前步骤的中间状态。
grade107=grade_agent107(completed107); manifest107={"schema":1,"policy":"script-or-model-v1","max_steps":6,"max_tool_calls":4,"cycle_window":4,"error_policy":ERROR_POLICY107}; digest107=hashlib.sha256(json.dumps(manifest107,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert grade107["task_success"] and grade107["safe"]  # 用受控断言验证关键不变量。
assert grade107["tool_calls"]==2 and grade107["stop"]=="finished"  # 用受控断言验证关键不变量。
assert len(digest107)==64 and manifest107["max_steps"]==6  # 用受控断言验证关键不变量。

## 面试总结

一套工程化 ReAct 回答是：**显式状态 → 结构化 action → 不可信 observation → 有界循环 → step/tool/token/deadline 预算 → cycle detector → 错误分类与重规划 → outcome/trajectory 联合评测**。Agent 的价值来自动态决策，可靠性来自模型外的确定性控制。

延伸阅读：[ReAct](https://arxiv.org/abs/2210.03629)、[Reflexion](https://arxiv.org/abs/2303.11366)、[Trustworthy Agents](https://www.anthropic.com/research/trustworthy-agents)。